### AML Deep Learning Model on The Elliptic Database
- Author: Andrea Colombo
- Course: GPU Computing - UniMi - A.A 2025/2026

### Environment Setup

The notebook was developed using Google CoLab so we made some assumptions about which libraries have to be installed and the hardware we are running on (T4 Runtime).
If you decide to run this notebook on a local environment you might have to change a couple lines here.

In [2]:
# The following two lines are meant to be used if the environment you are running the notebook
# in does not have torch_geometric and other dependencies installed.
# If you already have these, just comment the next two lines and setup the other env variables
# for your run
# This takes around 10 minutes to run
%pip install torch_geometric
%pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html

# This is the path where the dataset is going to be downloaded, if you are not
# running the notebook in colab you should likely change this
DATASET_PATH = '/content/data/elliptic'

# This variable should set to True only if you want to run the notebook with the CPU only
# Please note that doing so results in longer training times and garbage data in the GPU
# benchmark section.
ALLOW_CPU_FALLBACK = False

# This variable optionally disables plot, the report that comes along with this notebook
# already contains them so we might want to skip generating new ones on every run
ALLOW_PLOTS = True

# Fixed seed for the train/val split and model initialization, kept separate from the
# dataset's own temporal train/test split so runs are reproducible.
SEED = 42

### Notebook dependencies

All of these libraries are guaranteed to be available on colab after running the cell above

In [ ]:
import copy
import os
import platform
import functools
import torch
import time
import random
import numpy as np
import pandas as pd
import torch_geometric
from torch_geometric.nn import SAGEConv
from torch_geometric.datasets import EllipticBitcoinDataset
from torch_geometric.utils import subgraph
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
    roc_auc_score,
    average_precision_score,
)

if not torch.cuda.is_available() and not ALLOW_CPU_FALLBACK:
    raise RuntimeError(
        'No CUDA device found for this environment. This notebook expects a GPU runtime. '
        'If you want to run this notebook with the CPU anyway, change the value of '
        'ALLOW_CPU_FALLBACK to True (The GPU benchmark section will not provide '
        'meaningful results).'
    )

### Generic utilities for better development


In [4]:
class Logger:

    _t0 = time.perf_counter()

    def _prefix():
        return f"[{time.perf_counter() - Logger._t0:6.1f}s]"

    def log(msg):
        print('\033[92m' + f"[LOG]{Logger._prefix()} " + msg + '\033[0m')

    def warn(msg):
        print('\033[93m' + f"[WARN]{Logger._prefix()} " + msg + '\033[0m')

    def fatal(msg):
        print('\033[91m' + f"[ERR]{Logger._prefix()} " + msg + '\033[0m')

def checked_plot(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if ALLOW_PLOTS:
            return func(*args, **kwargs)
        else:
            Logger.warn(
                f"Skipped plot function {func.__name__}, set ENABLE_PLOTS = True "
                "if you want to visualize them"
            )
            return None
    return wrapper

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def get_elliptic_dataset():
    dataset = EllipticBitcoinDataset(root = DATASET_PATH)
    data = dataset[0]
    Logger.log(f"nodes: {data.num_nodes}, edges: {data.num_edges}, features: {data.num_features}")
    labeled = ((data.y == 0) | (data.y == 1)).sum().item()
    Logger.log(f"labeled nodes: {labeled}")
    return data

def make_splits(elliptic_data, val_fraction = 0.15, seed = 42):
    labels = elliptic_data.y
    labeled_mask = (labels == 0) | (labels == 1)

    train_mask = elliptic_data.train_mask.clone() & labeled_mask
    test_mask = elliptic_data.test_mask.clone() & labeled_mask

    train_idx = train_mask.nonzero(as_tuple = True)[0].cpu().numpy()
    train_labels = labels[train_idx].cpu().numpy()

    rng = np.random.default_rng(seed)
    val_idx = []
    for cls in (0, 1):
        cls_idx = train_idx[train_labels == cls]
        n_val = max(1, int(len(cls_idx) * val_fraction))
        val_idx.append(rng.choice(cls_idx, size = n_val, replace = False))
    val_idx = np.concatenate(val_idx)

    val_mask = torch.zeros_like(train_mask)
    val_mask[val_idx] = True
    train_mask[val_idx] = False

    Logger.log(
        f"Train nodes: {train_mask.sum().item()}, "
        f"Val nodes: {val_mask.sum().item()}, "
        f"Test nodes: {test_mask.sum().item()}"
    )

    return train_mask, val_mask, test_mask

def compute_pos_weight(labels, mask, device):
    y = labels[mask]
    num_licit = (y == 0).sum().item()
    num_illicit = (y == 1).sum().item()
    if num_illicit == 0:
        raise ValueError("Given mask contains zero illicit nodes, check the split.")

    pos_weight = torch.tensor([num_licit / num_illicit], dtype = torch.float32, device = device)
    # Logger.log(f"pos_weight: {pos_weight.item():.4f} (licit = {num_licit}, illicit = {num_illicit})")
    return pos_weight

elliptic_data = get_elliptic_dataset()

In [ ]:
class SkipGraphSAGE(torch.nn.Module):

    def __init__(self, in_chs, hidden_chs, dropout = 0.3):
        super(SkipGraphSAGE, self).__init__()
        self.sage_1 = SAGEConv(in_chs, hidden_chs)
        self.sage_2 = SAGEConv(hidden_chs, hidden_chs)
        self.bn_1 = torch.nn.BatchNorm1d(hidden_chs)
        self.bn_2 = torch.nn.BatchNorm1d(hidden_chs)
        self.dropout = dropout

        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(in_chs + hidden_chs, hidden_chs),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_chs, 1)
        )

    def embed(self, x, edge_index):
        h = self.sage_1(x, edge_index)
        h = self.bn_1(h).relu()
        h = torch.nn.functional.dropout(h, p = self.dropout, training = self.training)
        h = self.sage_2(h, edge_index)
        h = self.bn_2(h).relu()
        return h

    def forward(self, x, edge_index):
        h = self.embed(x, edge_index)
        z = torch.cat([x, h], dim = -1)
        return self.classifier(z).squeeze(-1)


In [7]:
set_seed(SEED)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
Logger.log(f"Using device = {device}")

elliptic_data_device = elliptic_data.to(device)

train_mask, val_mask, test_mask = make_splits(elliptic_data_device, val_fraction = 0.15, seed = SEED)
pos_weight = compute_pos_weight(elliptic_data_device.y, train_mask, device)

In [ ]:
IN_CHANNELS = elliptic_data.num_features
HIDDEN_CHANNELS = 128
DROPOUT = 0.3
NUM_EPOCHS = 300
LEARNING_RATE = 0.005
WEIGHT_DECAY = 1e-4
PATIENCE = 20
EVAL_EVERY = 5
PRINT_RUNNING_LOSS = True

# Message passing is restricted to non-test nodes for both training AND validation forward passes
non_test_mask = ~elliptic_data_device.test_mask
train_edge_index, _ = subgraph(
    non_test_mask,
    elliptic_data_device.edge_index,
    relabel_nodes = False,
    num_nodes = elliptic_data_device.num_nodes
)

def train_skip_graphsage(
    model, x, train_edge_index, train_mask, val_mask, y,
    optimizer, criterion, num_epochs, eval_every = 5, patience = 20, verbose = True
):
    best_val_ap = -1.0
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(x, train_edge_index)
        loss = criterion(logits[train_mask], y[train_mask])
        loss.backward()
        optimizer.step()

        if epoch % eval_every == 0:
            model.eval()
            with torch.no_grad():
                eval_logits = model(x, train_edge_index)
                val_probs = torch.sigmoid(eval_logits[val_mask]).cpu().numpy()
                val_ap = average_precision_score(y[val_mask].cpu().numpy(), val_probs)
            history.append({'epoch': epoch, 'train_loss': loss.item(), 'val_pr_auc': val_ap})

            if verbose:
                Logger.log(f"Epoch {epoch}: train loss = {loss.item():.4f}, val PR-AUC = {val_ap:.4f}")

            if val_ap > best_val_ap:
                best_val_ap = val_ap
                best_state = copy.deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += eval_every
                if epochs_without_improvement >= patience:
                    Logger.log(f"Early stopping at epoch {epoch}, best val PR-AUC = {best_val_ap:.4f}")
                    break

    model.load_state_dict(best_state)
    Logger.log(f"Restored best checkpoint (val PR-AUC = {best_val_ap:.4f})")
    return model, history

model = SkipGraphSAGE(IN_CHANNELS, HIDDEN_CHANNELS, dropout = DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE, weight_decay = WEIGHT_DECAY)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight = pos_weight)
y = elliptic_data_device.y.float()

model, train_history = train_skip_graphsage(
    model, elliptic_data_device.x, train_edge_index,
    train_mask, val_mask, y, optimizer, criterion,
    num_epochs = NUM_EPOCHS, eval_every = EVAL_EVERY, patience = PATIENCE,
    verbose = PRINT_RUNNING_LOSS
)

In [ ]:
def predict_probs(model, x, edge_index, mask):
    model.eval()
    with torch.no_grad():
        logits = model(x, edge_index)[mask]
    return torch.sigmoid(logits).cpu().numpy()

val_probs = predict_probs(model, elliptic_data_device.x, train_edge_index, val_mask)
val_true = y[val_mask].cpu().numpy()

precisions, recalls, thresholds = precision_recall_curve(val_true, val_probs)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-12)
best_idx = np.argmax(f1_scores[:-1])
BEST_THRESHOLD = thresholds[best_idx]

Logger.log(f"Selected threshold = {BEST_THRESHOLD:.4f} (val F1 = {f1_scores[best_idx]:.4f})")

In [14]:
test_probs_np = predict_probs(model, elliptic_data_device.x, elliptic_data_device.edge_index, test_mask)
y_true = y[test_mask].cpu().numpy()
test_preds = (test_probs_np >= BEST_THRESHOLD).astype(int)

roc_auc = roc_auc_score(y_true, test_probs_np)
pr_auc = average_precision_score(y_true, test_probs_np)

class_report = classification_report(y_true, test_preds, target_names=["Licit (0)", "Illicit (1)"], digits=4)
print(class_report)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")

cm = confusion_matrix(y_true, test_preds)
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives  (TP): {tp}")

In [ ]:
@checked_plot
def plot_roc_pr_curves(y_true, y_probs, roc_auc, pr_auc):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    precisions, recalls, _ = precision_recall_curve(y_true, y_probs)

    fig, axes = plt.subplots(1, 2, figsize = (12, 5), dpi = 150)

    axes[0].plot(fpr, tpr, color = '#1f77b4', label = f'ROC (AUC = {roc_auc:.3f})')
    axes[0].plot([0, 1], [0, 1], linestyle = '--', color = 'gray', label = 'Random')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Curve (Test Set)')
    axes[0].legend(loc = 'lower right')
    axes[0].grid(True, linestyle = '--', alpha = 0.4)

    axes[1].plot(recalls, precisions, color = '#d62728', label = f'PR (AUC = {pr_auc:.3f})')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall Curve (Test Set)')
    axes[1].legend(loc = 'lower left')
    axes[1].grid(True, linestyle = '--', alpha = 0.4)

    plt.tight_layout()
    plt.show()

plot_roc_pr_curves(y_true, test_probs_np, roc_auc, pr_auc)


### GPU Performance Evaluation

Wall clock evaluation of GPU vs CPU


In [ ]:
def get_cpu_name():
    try:
        with open('/proc/cpuinfo') as f:
            for line in f:
                if line.strip().startswith('model name'):
                    return line.split(':', 1)[1].strip()
    except FileNotFoundError:
        pass
    return platform.processor() or platform.uname().processor or 'unknown'

Logger.log(f"Python: {platform.python_version()}")
Logger.log(f"PyTorch: {torch.__version__} (built for CUDA {torch.version.cuda})")
Logger.log(f"PyTorch Geometric: {torch_geometric.__version__}")
Logger.log(f"CPU: {get_cpu_name()} ({os.cpu_count()} logical cores)")

if torch.cuda.is_available():
    gpu_props = torch.cuda.get_device_properties(0)
    Logger.log(
        f"GPU: {torch.cuda.get_device_name(0)} "
        f"({gpu_props.total_memory / 1e9:.1f} GB, compute capability {gpu_props.major}.{gpu_props.minor})"
    )
else:
    Logger.warn("No CUDA device available.")


In [ ]:
BENCH_TRAIN_ITERS = 15
BENCH_TRAIN_WARMUP = 5
BENCH_INFER_ITERS = 30
BENCH_INFER_WARMUP = 10

def _sync(device):
    if device.type == 'cuda':
        torch.cuda.synchronize()

def _timed_run(step_fn, device, n_iters, n_warmup):
    times = []
    for i in range(n_warmup + n_iters):
        _sync(device)
        t0 = time.perf_counter()
        step_fn()
        _sync(device)
        t1 = time.perf_counter()
        if i >= n_warmup:
            times.append(t1 - t0)
    return np.array(times)

def make_benchmark_assets(device):
    set_seed(SEED)
    bmodel = SkipGraphSAGE(IN_CHANNELS, HIDDEN_CHANNELS, dropout = DROPOUT).to(device)
    bx = elliptic_data.x.to(device)
    bei_train = train_edge_index.to(device)
    bei_full = elliptic_data.edge_index.to(device)
    bmask = train_mask.to(device)
    by = y.to(device)
    boptimizer = torch.optim.Adam(bmodel.parameters(), lr = LEARNING_RATE, weight_decay = WEIGHT_DECAY)
    bcriterion = torch.nn.BCEWithLogitsLoss(pos_weight = pos_weight.to(device))
    return bmodel, bx, bei_train, bei_full, bmask, by, boptimizer, bcriterion

bench_devices = ['cuda', 'cpu'] if torch.cuda.is_available() else ['cpu']
if 'cuda' not in bench_devices:
    Logger.warn("No CUDA device available, GPU benchmark skipped.")

bench_results = {}
for dev_name in bench_devices:
    dev = torch.device(dev_name)
    bmodel, bx, bei_train, bei_full, bmask, by, boptimizer, bcriterion = make_benchmark_assets(dev)

    if dev.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(dev)

    bmodel.train()
    def train_step():
        boptimizer.zero_grad()
        logits = bmodel(bx, bei_train)
        loss = bcriterion(logits[bmask], by[bmask])
        loss.backward()
        boptimizer.step()
    train_times = _timed_run(train_step, dev, BENCH_TRAIN_ITERS, BENCH_TRAIN_WARMUP)

    bmodel.eval()
    def infer_step():
        with torch.no_grad():
            bmodel(bx, bei_full)
    infer_times = _timed_run(infer_step, dev, BENCH_INFER_ITERS, BENCH_INFER_WARMUP)

    bench_results[dev_name] = {
        'train_mean_ms': train_times.mean() * 1000,
        'train_std_ms': train_times.std() * 1000,
        'train_min_ms': train_times.min() * 1000,
        'infer_mean_ms': infer_times.mean() * 1000,
        'infer_std_ms': infer_times.std() * 1000,
        'infer_min_ms': infer_times.min() * 1000,
    }
    if dev.type == 'cuda':
        bench_results[dev_name]['peak_mem_mb'] = torch.cuda.max_memory_allocated(dev) / 1e6

    Logger.log(
        f"[{dev_name}] train step: {train_times.mean()*1000:.2f} +/- {train_times.std()*1000:.2f} ms "
        f"(min {train_times.min()*1000:.2f} ms) | "
        f"inference: {infer_times.mean()*1000:.2f} +/- {infer_times.std()*1000:.2f} ms "
        f"(min {infer_times.min()*1000:.2f} ms)"
    )

bench_df = pd.DataFrame(bench_results).T
print(bench_df.round(3))

if 'cuda' in bench_results and 'cpu' in bench_results:
    train_speedup = bench_results['cpu']['train_min_ms'] / bench_results['cuda']['train_min_ms']
    infer_speedup = bench_results['cpu']['infer_min_ms'] / bench_results['cuda']['infer_min_ms']
    Logger.log(f"Training-step speedup (CPU / GPU, min-based): {train_speedup:.2f}x")
    Logger.log(f"Inference speedup (CPU / GPU, min-based): {infer_speedup:.2f}x")
    Logger.log(f"Peak GPU memory during benchmark: {bench_results['cuda']['peak_mem_mb']:.1f} MB")


In [27]:
@checked_plot
def plot_gpu_speedup(bench_results):
    if 'cuda' not in bench_results or 'cpu' not in bench_results:
        Logger.warn("Skipped GPU speedup plot, both cuda and cpu results are required.")
        return

    labels = ['Train step', 'Inference']
    cpu_means = [bench_results['cpu']['train_mean_ms'], bench_results['cpu']['infer_mean_ms']]
    cpu_stds = [bench_results['cpu']['train_std_ms'], bench_results['cpu']['infer_std_ms']]
    gpu_means = [bench_results['cuda']['train_mean_ms'], bench_results['cuda']['infer_mean_ms']]
    gpu_stds = [bench_results['cuda']['train_std_ms'], bench_results['cuda']['infer_std_ms']]

    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize = (6, 4.5), dpi = 150)
    ax.bar(x - width / 2, cpu_means, width, yerr = cpu_stds, capsize = 4, label = 'CPU', color = '#d62728')
    ax.bar(x + width / 2, gpu_means, width, yerr = gpu_stds, capsize = 4, label = 'GPU', color = '#1f77b4')
    ax.set_yscale('log')
    ax.set_ylabel('Wall-clock time (ms, log scale)')
    ax.set_title('CPU vs GPU: Full-Graph Training Step and Inference Latency')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    ax.grid(True, axis = 'y', linestyle = '--', alpha = 0.4)

    plt.tight_layout()
    plt.show()

plot_gpu_speedup(bench_results)
